# 02 — Precision, cancellation and money

## Usage and contracts
NearlyEqual uses min(4096 epsilon, sqrt(epsilon)) as the default absolute and relative tolerance. The explicit overload requires finite, non-negative tolerances; NaN is not equal to NaN, identical infinities are equal. Choose the tolerance according to the scale and the problem. KahanSum reduces some errors, does not remove cancellation and does not guarantee a correct result with non-finite values. InvoiceTotal and DecimalScale are System.Decimal specialisations, not Decimal128.
[Full reference](12-api-reference.ipynb), [checks on the six formats](11-consolidation.ipynb).
**JupyterSharp** kernel (C#) on the **.NET 11** runtime. Prerequisite: `dotnet build src/MathemaSharp.slnx -c Release`. From a terminal: `scripts/run-notebooks.ps1` (sets the roll-forward to runtime 11); in VS Code you need `DOTNET_ROLL_FORWARD=LatestMajor` and, as long as .NET 11 is RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

`decimal` has a 96-bit coefficient and scale 0–28: it is not IEEE Decimal128. No finite type represents 1/3 exactly.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
Console.WriteLine($"double: {0.1+0.2:R}; decimal: {0.1m+0.2m}");
Console.WriteLine($"ULP(1): {double.BitIncrement(1)-1:R}; Epsilon: {double.Epsilon:R}");
double[] values = {1e16, 1, 1, -1e16};
Console.WriteLine(NumericAlgorithms.Sum<double>(values));
Console.WriteLine(NumericAlgorithms.KahanSum<double>(values));
Console.WriteLine(Precision.NearlyEqual(0.1+0.2,0.3));

double: 0.30000000000000004; decimal: 0.3
ULP(1): 2.220446049250313E-16; Epsilon: 5E-324
0
2
True

In [ ]:
decimal[] lines = {0.005m,0.005m,0.005m};
foreach (var mode in new[] {MidpointRounding.ToEven,MidpointRounding.AwayFromZero})
    Console.WriteLine($"{mode}: lines={Precision.InvoiceTotal(lines,mode,true)}, total={Precision.InvoiceTotal(lines,mode,false)}");
Console.WriteLine($"Scale of 1.00m: {Precision.DecimalScale(1.00m)}");
Console.WriteLine(1m/3m);

ToEven: lines=0.00, total=0.02
AwayFromZero: lines=0.03, total=0.02
Scale of 1.00m: 2
0.3333333333333333333333333333

## Experiments
Reorder the data: Kahan is not a universally exact sum. Flip the sign of the amounts and choose an explicit policy. Absolute tolerances near zero, relative ones for large values. NaN is not equal to itself with ==. double.Epsilon is the smallest positive subnormal, not the distance between 1 and the next double.